# 05 — Bearing-Level Split and Sequence Preparation

Create fixed training, validation and test partitions by physical bearing, then prepare chronological five-recording sequences. The same physical bearing must not appear in more than one partition.

In [ ]:
# ============================================================
# STAGE 5: BEARING-LEVEL DATA SPLIT
# ============================================================

from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../data/processed")

feature_path = (
    PROCESSED_DIR /
    "paderborn_vibration_features.csv"
)

df = pd.read_csv(feature_path)

print("Dataset shape:", df.shape)
print("Unique bearings:", df["bearing"].nunique())

# ------------------------------------------------------------
# Fixed bearing-level split
# ------------------------------------------------------------

train_bearings = [
    # Healthy
    "K001", "K002", "K003", "K004",

    # Outer
    "KA01", "KA04", "KA05", "KA06",
    "KA07", "KA09", "KA15", "KA16", "KA30",

    # Inner
    "KI01", "KI04", "KI05", "KI07",
    "KI08", "KI14", "KI16", "KI17",

    # Both
    "KB23"
]

val_bearings = [
    "K005",
    "KA03", "KA08",
    "KI03",
    "KB24"
]

test_bearings = [
    "K006",
    "KA22",
    "KI18", "KI21",
    "KB27"
]

# ------------------------------------------------------------
# Check that every bearing appears exactly once
# ------------------------------------------------------------

all_split_bearings = (
    train_bearings
    + val_bearings
    + test_bearings
)

assert len(all_split_bearings) == 32
assert len(set(all_split_bearings)) == 32

dataset_bearings = set(df["bearing"].unique())

assert (
    set(all_split_bearings) == dataset_bearings
), "Split does not contain exactly the dataset bearings."

# ------------------------------------------------------------
# Create split DataFrames
# ------------------------------------------------------------

train_df = df[
    df["bearing"].isin(train_bearings)
].copy()

val_df = df[
    df["bearing"].isin(val_bearings)
].copy()

test_df = df[
    df["bearing"].isin(test_bearings)
].copy()

print("\nTrain rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

In [ ]:
# ============================================================
# VERIFY SPLIT COMPOSITION
# ============================================================

def show_split_summary(name, split_df):

    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")

    print("Rows:", len(split_df))
    print("Unique bearings:", split_df["bearing"].nunique())

    print("\nFault-location:")
    print(
        split_df["fault_location"]
        .value_counts()
        .sort_index()
    )

    print("\nFault-class:")
    print(
        split_df["fault_class"]
        .value_counts()
        .sort_index()
    )

    print("\nBinary label:")
    print(
        split_df["binary_label"]
        .value_counts()
        .sort_index()
    )


show_split_summary("TRAIN", train_df)
show_split_summary("VALIDATION", val_df)
show_split_summary("TEST", test_df)

In [ ]:
# ============================================================
# VERIFY BEARING ASSIGNMENT
# ============================================================

print("TRAIN BEARINGS:")
print(train_bearings)

print("\nVALIDATION BEARINGS:")
print(val_bearings)

print("\nTEST BEARINGS:")
print(test_bearings)

In [ ]:
# ============================================================
# DEFINE FEATURE COLUMNS
# ============================================================

metadata_columns = [
    "file",
    "bearing",
    "operating_condition",
    "measurement_number",
    "fault_location",
    "fault_class",
    "binary_label"
]

feature_columns = [
    col
    for col in df.columns
    if col not in metadata_columns
]

print("Total extracted features:", len(feature_columns))
print(feature_columns)

In [ ]:
# ============================================================
# TRAINING-ONLY FEATURE CORRELATION
# ============================================================

train_corr = train_df[feature_columns].corr()

correlation_pairs = []

for i in range(len(feature_columns)):
    for j in range(i + 1, len(feature_columns)):

        f1 = feature_columns[i]
        f2 = feature_columns[j]

        corr = train_corr.loc[f1, f2]

        if abs(corr) >= 0.90:
            correlation_pairs.append({
                "feature_1": f1,
                "feature_2": f2,
                "correlation": corr
            })

train_high_corr = pd.DataFrame(
    correlation_pairs
)

if len(train_high_corr) > 0:
    train_high_corr = (
        train_high_corr
        .sort_values(
            "correlation",
            key=lambda x: x.abs(),
            ascending=False
        )
    )

print(
    "Highly correlated pairs in TRAINING data:"
)

display(train_high_corr)

In [ ]:
# ============================================================
# REMOVE CLEARLY REDUNDANT FEATURES
# ============================================================

redundant_features = [
    "rms",
    "spectral_rms",
    "peak_to_peak",
    "spectral_entropy"
]

reduced_features = [
    feature
    for feature in feature_columns
    if feature not in redundant_features
]

print("Original number of features:", len(feature_columns))
print("Removed features:", redundant_features)
print("Remaining features:", len(reduced_features))

print("\nRemaining features:")
for feature in reduced_features:
    print("-", feature)

In [ ]:
# ============================================================
# CHECK REDUCED FEATURE CORRELATION
# ============================================================

reduced_corr = train_df[reduced_features].corr()

remaining_high_corr = []

for i in range(len(reduced_features)):
    for j in range(i + 1, len(reduced_features)):

        f1 = reduced_features[i]
        f2 = reduced_features[j]

        corr = reduced_corr.loc[f1, f2]

        if abs(corr) >= 0.90:
            remaining_high_corr.append({
                "feature_1": f1,
                "feature_2": f2,
                "correlation": corr
            })

remaining_high_corr_df = pd.DataFrame(
    remaining_high_corr
)

if len(remaining_high_corr_df) > 0:
    remaining_high_corr_df = (
        remaining_high_corr_df
        .sort_values(
            "correlation",
            key=lambda x: x.abs(),
            ascending=False
        )
    )

print(
    "Remaining highly correlated pairs:"
)

display(remaining_high_corr_df)

In [ ]:
# ============================================================
# STAGE 5A: TRAINING-ONLY FEATURE RANKING
# STAGE 1: HEALTHY vs DAMAGED
# ============================================================

from sklearn.feature_selection import f_classif

# Training data only
X_train_features = train_df[reduced_features]
y_train_binary = train_df["binary_label"]

# ANOVA F-score
f_scores, p_values = f_classif(
    X_train_features,
    y_train_binary
)

feature_ranking = pd.DataFrame({
    "feature": reduced_features,
    "f_score": f_scores,
    "p_value": p_values
})

feature_ranking = (
    feature_ranking
    .sort_values(
        "f_score",
        ascending=False
    )
    .reset_index(drop=True)
)

print("Training-only ANOVA ranking:")
display(
    feature_ranking
)

In [ ]:
# ============================================================
# FEATURE SETS FOR MODEL COMPARISON
# ============================================================

top_12_features = (
    feature_ranking
    .head(12)["feature"]
    .tolist()
)

top_15_features = (
    feature_ranking
    .head(15)["feature"]
    .tolist()
)

all_22_features = reduced_features.copy()

print("TOP 12 FEATURES:")
for i, feature in enumerate(top_12_features, start=1):
    print(i, feature)

print("\nTOP 15 FEATURES:")
for i, feature in enumerate(top_15_features, start=1):
    print(i, feature)

print("\nALL 22 FEATURES:")
for i, feature in enumerate(all_22_features, start=1):
    print(i, feature)

In [ ]:
# ============================================================
# TOP FEATURES
# ============================================================

print("Top 15 features:")
display(
    feature_ranking.head(15)
)

In [ ]:
# ============================================================
# STAGE 5B: SEQUENCE CONSTRUCTION
# ============================================================

import numpy as np


def create_sequences(
    data,
    features,
    sequence_length=5,
    target_column="binary_label",
    require_consecutive=True
):
    """
    Create sliding-window sequences separately for each
    bearing and operating condition.

    If require_consecutive=True, windows are created only when
    measurement numbers are consecutive. This prevents the
    missing KA08 measurement from creating a false sequence.
    """

    X_sequences = []
    y_sequences = []

    sequence_metadata = []

    for (bearing, condition), group in data.groupby(
        ["bearing", "operating_condition"]
    ):

        group = (
            group
            .sort_values("measurement_number")
            .reset_index(drop=True)
        )

        # ----------------------------------------------------
        # Split into consecutive runs if a measurement is missing
        # ----------------------------------------------------

        if require_consecutive:

            measurement_numbers = (
                group["measurement_number"]
                .to_numpy()
            )

            run_breaks = np.where(
                np.diff(measurement_numbers) != 1
            )[0] + 1

            runs = np.split(
                np.arange(len(group)),
                run_breaks
            )

        else:

            runs = [
                np.arange(len(group))
            ]

        # ----------------------------------------------------
        # Create sequences within each consecutive run
        # ----------------------------------------------------

        for run_indices in runs:

            run = group.iloc[run_indices]

            if len(run) < sequence_length:
                continue

            X_values = run[features].to_numpy(
                dtype=np.float32
            )

            y_values = run[target_column].to_numpy()

            measurement_values = (
                run["measurement_number"]
                .to_numpy()
            )

            for i in range(
                len(run) - sequence_length + 1
            ):

                X_seq = X_values[
                    i:i + sequence_length
                ]

                y_value = y_values[
                    i + sequence_length - 1
                ]

                X_sequences.append(X_seq)
                y_sequences.append(y_value)

                sequence_metadata.append({
                    "bearing": bearing,
                    "operating_condition": condition,
                    "sequence_start": int(
                        measurement_values[i]
                    ),
                    "sequence_end": int(
                        measurement_values[
                            i + sequence_length - 1
                        ]
                    )
                })

    return (
        np.asarray(X_sequences, dtype=np.float32),
        np.asarray(y_sequences),
        pd.DataFrame(sequence_metadata)
    )

In [ ]:
# ============================================================
# STAGE 1: HEALTHY vs DAMAGED
# ============================================================

SEQUENCE_LENGTH = 5

feature_sets = {
    "top_12": top_12_features,
    "top_15": top_15_features,
    "all_22": all_22_features
}

stage1_sequences = {}

for set_name, features in feature_sets.items():

    X_train, y_train, meta_train = create_sequences(
        train_df,
        features,
        sequence_length=SEQUENCE_LENGTH,
        target_column="binary_label"
    )

    X_val, y_val, meta_val = create_sequences(
        val_df,
        features,
        sequence_length=SEQUENCE_LENGTH,
        target_column="binary_label"
    )

    X_test, y_test, meta_test = create_sequences(
        test_df,
        features,
        sequence_length=SEQUENCE_LENGTH,
        target_column="binary_label"
    )

    stage1_sequences[set_name] = {
        "features": features,
        "X_train": X_train,
        "y_train": y_train,
        "meta_train": meta_train,
        "X_val": X_val,
        "y_val": y_val,
        "meta_val": meta_val,
        "X_test": X_test,
        "y_test": y_test,
        "meta_test": meta_test
    }

    print(
        f"\n{set_name.upper()}"
    )
    print("Train:", X_train.shape, y_train.shape)
    print("Val:  ", X_val.shape, y_val.shape)
    print("Test: ", X_test.shape, y_test.shape)

In [ ]:
# ============================================================
# STAGE 2: DAMAGE LOCATION
# Outer vs Inner vs Both
# ============================================================

stage2_train_df = train_df[
    train_df["binary_label"] == 1
].copy()

stage2_val_df = val_df[
    val_df["binary_label"] == 1
].copy()

stage2_test_df = test_df[
    test_df["binary_label"] == 1
].copy()

stage2_sequences = {}

for set_name, features in feature_sets.items():

    X_train, y_train, meta_train = create_sequences(
        stage2_train_df,
        features,
        sequence_length=SEQUENCE_LENGTH,
        target_column="fault_class"
    )

    X_val, y_val, meta_val = create_sequences(
        stage2_val_df,
        features,
        sequence_length=SEQUENCE_LENGTH,
        target_column="fault_class"
    )

    X_test, y_test, meta_test = create_sequences(
        stage2_test_df,
        features,
        sequence_length=SEQUENCE_LENGTH,
        target_column="fault_class"
    )

    stage2_sequences[set_name] = {
        "features": features,
        "X_train": X_train,
        "y_train": y_train,
        "meta_train": meta_train,
        "X_val": X_val,
        "y_val": y_val,
        "meta_val": meta_val,
        "X_test": X_test,
        "y_test": y_test,
        "meta_test": meta_test
    }

    print(
        f"\n{set_name.upper()}"
    )
    print("Train:", X_train.shape, y_train.shape)
    print("Val:  ", X_val.shape, y_val.shape)
    print("Test: ", X_test.shape, y_test.shape)

In [ ]:
# ============================================================
# STAGE 5C: TRAINING-ONLY SEQUENCE SCALING
# ============================================================

from sklearn.preprocessing import StandardScaler

scaled_stage1 = {}
scaled_stage2 = {}

# ------------------------------------------------------------
# Stage 1 scaling
# ------------------------------------------------------------

for set_name, data_dict in stage1_sequences.items():

    X_train = data_dict["X_train"]
    X_val = data_dict["X_val"]
    X_test = data_dict["X_test"]

    n_samples, timesteps, n_features = X_train.shape

    scaler = StandardScaler()

    # Fit ONLY on training data
    scaler.fit(
        X_train.reshape(-1, n_features)
    )

    X_train_scaled = scaler.transform(
        X_train.reshape(-1, n_features)
    ).reshape(X_train.shape)

    X_val_scaled = scaler.transform(
        X_val.reshape(-1, n_features)
    ).reshape(X_val.shape)

    X_test_scaled = scaler.transform(
        X_test.reshape(-1, n_features)
    ).reshape(X_test.shape)

    scaled_stage1[set_name] = {
        "scaler": scaler,
        "X_train": X_train_scaled.astype(np.float32),
        "y_train": data_dict["y_train"],
        "X_val": X_val_scaled.astype(np.float32),
        "y_val": data_dict["y_val"],
        "X_test": X_test_scaled.astype(np.float32),
        "y_test": data_dict["y_test"],
        "meta_train": data_dict["meta_train"],
        "meta_val": data_dict["meta_val"],
        "meta_test": data_dict["meta_test"]
    }


# ------------------------------------------------------------
# Stage 2 scaling
# ------------------------------------------------------------

for set_name, data_dict in stage2_sequences.items():

    X_train = data_dict["X_train"]
    X_val = data_dict["X_val"]
    X_test = data_dict["X_test"]

    n_samples, timesteps, n_features = X_train.shape

    scaler = StandardScaler()

    # Fit ONLY on training data
    scaler.fit(
        X_train.reshape(-1, n_features)
    )

    X_train_scaled = scaler.transform(
        X_train.reshape(-1, n_features)
    ).reshape(X_train.shape)

    X_val_scaled = scaler.transform(
        X_val.reshape(-1, n_features)
    ).reshape(X_val.shape)

    X_test_scaled = scaler.transform(
        X_test.reshape(-1, n_features)
    ).reshape(X_test.shape)

    scaled_stage2[set_name] = {
        "scaler": scaler,
        "X_train": X_train_scaled.astype(np.float32),
        "y_train": data_dict["y_train"],
        "X_val": X_val_scaled.astype(np.float32),
        "y_val": data_dict["y_val"],
        "X_test": X_test_scaled.astype(np.float32),
        "y_test": data_dict["y_test"],
        "meta_train": data_dict["meta_train"],
        "meta_val": data_dict["meta_val"],
        "meta_test": data_dict["meta_test"]
    }

print("Scaling completed for all feature configurations.")

In [ ]:
# ============================================================
# VERIFY SCALED SEQUENCES
# ============================================================

print("=" * 70)
print("STAGE 1 - SCALED SEQUENCES")
print("=" * 70)

for set_name, data_dict in scaled_stage1.items():

    print(
        f"{set_name}: "
        f"Train {data_dict['X_train'].shape}, "
        f"Val {data_dict['X_val'].shape}, "
        f"Test {data_dict['X_test'].shape}"
    )


print("\n" + "=" * 70)
print("STAGE 2 - SCALED SEQUENCES")
print("=" * 70)

for set_name, data_dict in scaled_stage2.items():

    print(
        f"{set_name}: "
        f"Train {data_dict['X_train'].shape}, "
        f"Val {data_dict['X_val'].shape}, "
        f"Test {data_dict['X_test'].shape}"
    )

In [ ]:
# ============================================================
# REMAP STAGE 2 LABELS
# ============================================================

stage2_label_map = {
    1: 0,   # outer
    2: 1,   # inner
    3: 2    # both
}

for set_name, data_dict in scaled_stage2.items():

    data_dict["y_train"] = np.array([
        stage2_label_map[int(y)]
        for y in data_dict["y_train"]
    ], dtype=np.int32)

    data_dict["y_val"] = np.array([
        stage2_label_map[int(y)]
        for y in data_dict["y_val"]
    ], dtype=np.int32)

    data_dict["y_test"] = np.array([
        stage2_label_map[int(y)]
        for y in data_dict["y_test"]
    ], dtype=np.int32)

print("Stage 2 labels remapped:")
print("0 = outer")
print("1 = inner")
print("2 = both")

In [ ]:
# ============================================================
# SAVE PREPARED SEQUENCES FOR MODEL TRAINING
# ============================================================

import joblib

stage1_path = (
    PROCESSED_DIR /
    "stage1_scaled_sequences.joblib"
)

stage2_path = (
    PROCESSED_DIR /
    "stage2_scaled_sequences.joblib"
)

feature_ranking_path = (
    PROCESSED_DIR /
    "training_feature_ranking.csv"
)

joblib.dump(
    scaled_stage1,
    stage1_path
)

joblib.dump(
    scaled_stage2,
    stage2_path
)

feature_ranking.to_csv(
    feature_ranking_path,
    index=False
)

print("Saved Stage 1:", stage1_path.resolve())
print("Saved Stage 2:", stage2_path.resolve())
print(
    "Saved feature ranking:",
    feature_ranking_path.resolve()
)